<p><small><small>This Notebook is made available subject to the licence and terms set out in the <a href = "http://www.github.com/google-deepmind/ai-foundations">AI Research Foundations Github README file</a>.</small></small></p>

<img src="https://storage.googleapis.com/dm-educational/assets/ai_foundations/GDM-Labs-banner-image-C7-white-bg.png">

# Lab: Prepare your Data for Building a Conversational Model

<a href='https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_8/gdm_lab_8_8_conversation_part_1.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg' alt='Open In Colab'/></a>

2-4 hours (depending on the amount of data cleaning necessary)

In this first lab, you will load, clean and format the data for training a conversational model. At this stage, you should have completed previous activities from 08 Capstone: Develop Your Model for Real-World Impact. Specifically you should have:

1. Defined your conversational task,
2. Downloaded or constructed your dataset with prompts and responses,
3. Documented your dataset with a Data Card, and
4. Assessed the impact of your conversational model.

In this and the next lab, you will build on the activities to turn your project plan into a working conversational model.

Similarly, as in the previous activities, this lab also contains a worked example for every step. Consider the code for the worked example as a template for your own implementation. Note that some steps may not be applicable to your project. You may have to complete additional steps, so treat the template in this and the next lab just as a starting point and feel free to add or delete code cells to this notebook as you see fit.

## Overview

In this first part of implementing your capstone project, you will focus on preparing your data for training a conversational model.

### What you will learn

By the end of this first part of the capstone project, you will be able to:

* Clean and preprocess text data to prepare quality inputs for fine-tuning instruction-based conversational models.

* Split datasets into training and test sets.

* Format the conversational data so that it can be used to fine-tune an LLM.

### Tasks

In this lab, you will:

* Load the dataset that you prepared in previous activities.

* Clean individual datapoints, if necessary.

* Split the dataset into training and test sets.

* Format the data and prepare it for LLM fine-tuning.

* Save the dataset in a format that is compatible with the machine learning pipeline.

The end result of this lab will be a processed dataset that you can use in the second part to train and evaluate your conversational model.

### What you will use

This lab builds on materials from several previous courses. If you would like to refresh your knowledge on any of the following topics, you can find the relevant materials in the following courses:

- **General knowledge on LLMs and transformers** (Courses 01 Build Your Own Small Language Model, 03 Design and Train Neural Networks, and 04 Discover The Transformer Architecture).

- **Data preparation** techniques (Course 02 Represent Your Text Data).

- **Formatting data for fine-tuning** (Course 05 Fine-Tune Your Model).

- **Splitting data** techniques (Course 03 Training a Neural Network).


## How to use Google Colaboratory (Colab)

Google Colaboratory (also known as Google Colab) is a platform that allows you to run Python code in your browser. The code is written in cells that are executed on a remote server.

To run a cell, hover over a cell, and click on the `run` button to its left. The run button is the circle with the triangle (▶). Alternatively, you can also click on a cell and use the keyboard combination Ctrl+Return (or ⌘+Return if you are using a Mac).

To try this out, run the following cell. This should print today's day of the week below it.

In [ ]:
from datetime import datetime
print(f"Today is {datetime.today():%A}.")

Note that the order in which you run the cells matters. When you are working through a lab, make sure to always run all cells in order. Otherwise, the code might not work. If you take a break while working on a lab, Colab may disconnect you and in that case, you have to execute all cells again before  continuing your work. To make this easier, you can select the cell you are currently working on and then choose __Runtime → Run before__  from the menu above (or use the keyboard combination Ctrl/⌘ + F8). This will re-execute all cells before the current one.

### Using Colab with a GPU

Follow these steps to run the activities in this lab on a GPU:

1.  In the top menu bar, click on **Runtime**.
2.  Select **Change runtime type** from the dropdown menu.
3.  In the pop-up window under **Hardware Accelerator**, select **GPU** (usually listed as `T4 GPU`).
5.  Click **Save**.

Your Colab session will now restart with GPU access.

Note that access to GPUs is limited and at times, you may not be able to run this lab on a GPU. All activities will still work but they will run slower and you will have to wait longer for some of the cells to finish running.


## Imports

The following cell contains imports for a range of libraries that you may use for preparing your dataset. Depending on your implementation, you may need to import additional libraries.

In [ ]:
# Standard library imports.
import io # Input/output operations.
import json # JSON data handling.
import os # Operating system interface.
import re # Regular expressions for text processing.
import unicodedata # Unicode character database.
from textwrap import fill # Text wrapping utilities.
from typing import Any, Dict, Tuple, List

# Third-party data and utility libraries.
import numpy as np # Numerical computing.
import pandas as pd # Data manipulation and analysis.
from tqdm import tqdm # Progress bars.

from sklearn.model_selection import train_test_split # Data splitting utils.
import matplotlib.pyplot as plt # Plotting and visualisation.

# Deep learning and AI libraries.
import jax.numpy as jnp # JAX numerical computing.
import keras # High-level neural networks API.
import keras_nlp # Keras NLP extensions.

# Google Colab specific.
from google.colab import drive # Access to Google Drive.
from google.colab import userdata # Access to Colab secrets.

## Load your data

As a first step, you will have to load the dataset that you created in the previous activities, so that you can preprocess your data. One method to do this is to upload the dataset to Google Drive and then connect your Google Drive to Colab.

### Connect Colab with Google Drive

The following cell mounts your Google Drive so that you can access files by adding `/content/drive/MyDrive/` to the beginning of your file path. Running this cell will open a dialog that asks you whether you want to connect Colab to Google Drive. Follow the instructions and enable all permissions, so that Colab can read and write files from Google Drive.


In [ ]:
drive.mount('/content/drive')

### Activity 1: Load and inspect your data


------
> 💻 **Your task:**
>
> Load and inspect your dataset by adding your code to the subsequent cells:
>
> 1. **Define the path to your dataset**: Set the path to your own dataset file path.
>
> 2. **Choose a data loading method** to match your data format:
>    - For CSV files: `pd.read_csv(data, encoding="utf-8")`.
>    - For JSON files: Use `json.load()` or `pd.read_json(data)`.
>    - For JSONL files: Use `pd.read_json(data, lines=True)`.
>    - For API data: Implement your own loading function.
>
> 3. **Verify your data structure**: After loading, check that:
>    - Your prompts are in an appropriate column.
>    - Your responses are appropriately formatted.
>    - The encoding is preserved (especially for non-English text).
>
> 4. **Inspect the data**: Use `df.head()`, `df.info()`, and `df.columns` to understand your dataset structure and size.
>    - Identify the prompt and response columns.
>
>
> The goal of these steps is to bring all records into a variable (a Pandas `DataFrame`) so that you can explore the structure, check the columns, preview a few rows, and confirm the encoding, before moving on to transformation for LoRA training.
------


Before you can transform your data into a LoRA instruction-tuning format, the data need to be loaded first.

The goal of this step is to bring all records into a variable (a Pandas `DataFrame`) so that you can explore the structure, check the columns, preview a few rows, and confirm the encoding, before moving on to transformation for LoRA training.

#### Worked example: Load and inspect your data





```python
# Open the file from Google Drive.
filepath = "/content/drive/MyDrive/africa_galore_qa_food.jsonl"

# Read with pandas.
df = pd.read_json(filepath, lines=True)
# Other file-type examples:
# df = pd.read_json(filepath, orient="records")
# df = pd.read_csv(filepath, encoding="utf-8")
# df = json.load(open(filepath))
# df = load_from_api()

# Take a quick look at the first few rows.
display(df.head())

print("Columns:", df.columns.tolist())
print("Number of rows:", len(df))
```

#### Your implementation


In [ ]:
# Add your code here.


### Activity 2: Clean the data

------
> 💻 **Your task:**
>
> Clean and prepare your text data for fine-tuning. This could, for example, involve the following steps, but the exact cleaning procedure will depend on where you sourced your data from and what kind of format it is in.
>
> 1. **Assess your data quality**: Examine your text data for:
>    - HTML tags, markup, or special formatting.
>    - Non-text characters (emojis, symbols, special Unicode).
>    - Inconsistent encoding or corrupted characters.
>    - Excessive whitespace or formatting issues.
>
> 2. **Apply appropriate cleaning functions**:
>    - Use `clean_html()` to remove HTML tags and entities.
>    - Use `clean_unicode()` to remove emojis and non-text symbols.
>    - Add custom cleaning functions if needed for your domain.
>
> 3. **Preserve original data**: Keep a copy of your original response before cleaning. In the worked example, this is done by keeping the original response in a new column:
>    ```python
>    df["response_original"] = df["response"].astype(str)
>    ```
>
> 4. **Preview cleaning results**: Compare before/after examples to ensure:
>    - Important content is preserved.
>    - Unwanted elements are removed.
>    - Text remains readable and coherent.
>
> 5. **Decide on final text column**: Choose whether to:
>    - Replace original text with cleaned version.
>    - Keep both versions for comparison.
>    - Apply different levels of cleaning for different purposes.
>
> **Quality check**: Examine several examples to verify that cleaning improves data quality without losing essential information.
>
------

Refer to Course 02 Represent Your Text Data for examples on how to adjust and clean your data. The following code provides utility functions from Course 02 Represent Your Text Data to clean texts. You may want to add your own cleaning functions or modify the ones below depending on what kind of cleaning your data requires.

In [ ]:
def clean_html(text: str) -> str:
    """Strip basic HTML markup and common entities from a string.

    The funcion does **not** attempt full HTML parsing; for more complex markup
    consider ``BeautifulSoup`` or ``html.unescape``.

    Args:
      text: The text string that may contain HTML tags or entities.

    Returns:
      A cleaned string with tags stripped and the entities '&nbsp;', '&amp;',
        '&lt;' and '&gt;' converted to ' ', '&', '<' and '>'.
    """

    # Remove HTML tags.
    text = re.sub(r"<.*?>", "", text)

    # Replace HTML entities.
    text = re.sub("&nbsp;", " ", text)  # Replace non-breaking space.
    text = re.sub("&amp;", "&", text)  # Replace "&amp;" with "&".
    text = re.sub("&lt;", "<", text)  # Replace "&lt;" with "<".
    text = re.sub("&gt;", ">", text)  # Replace "&gt;" with ">".

    return text


def clean_unicode(text: str) -> str:
    """Removes non-text unicode characters from a string.

    Args:
      text: The original text which may contain special characters.

    Returns:
      The input text string with emojis and other non-text symbols removed.
    """

    categories_to_keep = {"L", "N", "P"}  # L=letters, N=numbers, P=punctuation.

    keep = []
    for ch in text:
        do_keep = ch.isspace()  # Preserve spaces.
        if not do_keep:
            for category in categories_to_keep:
                if unicodedata.category(ch).startswith(category):
                    do_keep = True
                    break
        if do_keep:
            keep.append(ch)
    return "".join(keep)

You can now apply the data cleaning functions to your data.

#### Worked example: Cleaning data in a DataFrame




```python
# Clean the target column (e.g., df["response"]) and preview before/after.
# For the time being, keep an original copy for reference.
df["response_original"] = df["response"].astype(str)
df["response_clean"] = (
    df["response"].map(clean_html).map(clean_unicode)
)

# Quick preview.
display(df.head())
```

#### Your implementation


In [ ]:
# Add your code for cleaning texts in your dataset here.


You may notice that, after cleaning the descriptions, some rows contain empty values. This typically occurs when the original description consisted primarily of noise patterns such as `<br><br>`, which are removed during the cleaning process. This highlights an important aspect of working with real-world data: datasets often contain imperfections that must be addressed as part of preprocessing.

To handle this, you can implement a `drop_empty_rows` function, as shown in the worked example. This function removes all rows where the `description` field is empty after cleaning, ensuring that only meaningful text is retained for downstream training and evaluation.


#### Worked example: Removing examples with empty values




```python
def drop_empty_rows(
    df: pd.DataFrame, column: str = "response_clean"
) -> pd.DataFrame:
    """Removes DataFrame rows where the specified column is empty.

    Args:
      df: The input DataFrame.
      column: Name of the column to inspect.

    Returns:
      A new DataFrame with rows containing empty or whitespace-only
      values in the specified column removed.
    """

    # Ensure the column is treated as string and trim whitespace.
    mask = df[column].astype(str).str.strip() != ""

    return df.loc[mask].reset_index(drop=True)


# Apply after response_clean has been created.
df = drop_empty_rows(df, column="response_clean")

# Quick preview.
display(df.head())

print("Columns:", df.columns.tolist())
print("Number of rows:", len(df))
```

#### Your implementation


In [ ]:
# If you need to remove values that are empty or perform other post-processing
# steps, add your code here.


After completing the text cleaning and row filtering steps, the cleaned descriptions are ready to replace the original raw text. In the worked example, the original `response` column is no longer needed and is removed to avoid confusion or accidental reuse. The cleaned version, stored in `response_clean`, is then renamed to `response` so that downstream processing and modeling steps can work with a single, consistently named column.

#### Worked example: Replace response field with the cleaned response filed



```python
# Replace the original response with the cleaned version
df = df.drop(columns=["response"]).rename(
    columns={"response_clean": "response"}
)

# Quick check
display(df.head())
print("Columns:", df.columns.tolist())
```


#### Your implementation


In [ ]:
# If you would like to replace fields in your original dataframe, add your
# code here.


### Activity 3: Split the original data


------
> 💻 **Your task:**
>
>  Split training data into train/test sets:
>
>  1. If you do not have a separate test dataset then you can split your working dataset before fine-tuning. For example:
>     - You can use the `train_test_split` function to split your data into a training split and a test split.
>       ```python
>       from sklearn.model_selection import train_test_split
>       df_train, df_test = train_test_split(df, test_size=0.2)
>       ```
>
>  - **Important considerations**:
>    - 80/20 is one of the most common train/test split ratios.
>    - **Performance warning**: Gemma3-1B may perform poorly on completely unknown test examples if your training dataset is small.
>    - For small datasets, consider using more training data and smaller test sets.
>    - Ensure evaluation data reflects your actual use case and domain.
------

The conversation pathway presents more complex and subjective evaluation challenges compared to other pathways, such as classification. Conversational responses are more open-ended, making performance assessment far more challenging and requiring primarily qualitative evaluation methods such as human assessment, conversational quality checks, and manual spot-checking for accuracy, helpfulness, engagement, and tone.

You may recall from Course 03 Design and Train Neural Networks that it is standard to split datasets into three splits: one for training, one for validation, and one for testing. In this capstone you will build a model that produces very open-ended answers, however, you will likely not be able to use any automatic metrics. Having a validation set will likely be of limited value, and the worked example splits the data only into a training and test split. You may want to do the same for your dataset.

#### Worked example: Split the data




```python
# Perform an 80%/20% train-test split.
df_train, df_test = train_test_split(df, test_size=0.2)
```

#### Your implementation


In [ ]:
# Add your code for splitting your dataset here.


### Activity 4: Format the data


------
> 💻 **Your task:**
>
> Convert your Q&A pairs in the training and test set to the format required for fine-tuning a model. Since at this point, your data should already consist of prompts and responses, this will likely be a very straightforward step where you simply have to extract the correct fields in your dataframe and store them in a dictionary with the `"input"` and `"output"` keys.
>
> 1. **Understand the data transformation**: The `to_instruction_record` function converts your generated Q&A pairs:
>    - Takes `row["prompt"]` (user questions about African food/culture for the example) as input.
>    - Takes `row["response"]` (informative answers) as output.
>    - Creates the standard `{"input": question, "output": answer}` format required for fine-tuning an LLM.
>
> 2. **Review the Q&A pair structure**: Verify that your data looks reasonable. In the worked example, the examples would, for example, look as follows.
>    - **Input examples**: 'What is one popular dish from Nigeria?', 'Tell me about signature food from Kenya'.
>    - **Output examples**: 'This dish captures the authentic flavor and spirit of Nigeria. Jollof Rice — A vibrant rice dish...'.
>    - **Quality check**: Check that answers are informative and contextual.
>
> 3. **Verify the transformation**: Run the code to convert your Q&A pairs and inspect the results:
>    - Check that the first few records show proper question-answer formatting.
>
> 4. **Save your training and test data**: Add code to save your formatted data as `/content/drive/MyDrive/conversational_training_data.jsonl` and `/content/drive/MyDrive/conversational_test_data.jsonl`:
>    - Each line should contain one complete prompt-response pair in instruction-tuning format.
>
>
> **Verification**: Perform final checks and verify that your JSONL file contains  prompt-response pairs as expected, with each record having 'input' (user question) and 'output' (informative response) fields.
>
------


#### Worked example: Format the conversational data



```python
def to_instruction_record(row: Dict[str, Any]) -> Dict[str, str]:
    """Transform a single row of a dataset into instruction-tuning format.

    Args:
      row: A single row from a Pandas DataFrame.

    Returns:
      dict: A dictionary with "input" and "output" as keys.
    """

    return {
        "input": row["prompt"],
        "output": row["response"],
    }


# Apply transformation to df_train and df_test.
instruction_records_train = [
    to_instruction_record(e) for _, e in df_train.iterrows()
]
instruction_records_test = [
    to_instruction_record(e) for _, e in df_test.iterrows()
]


# Preview a sample.
print(json.dumps(instruction_records_train[0], ensure_ascii=False, indent=2))

# Save to JSONL.
with open(
    "/content/drive/MyDrive/conversational_training_data.jsonl", "w", encoding="utf-8"
) as f:
    for rec in instruction_records_train:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(
    f"\nSaved {len(instruction_records_train)} instruction-tuning records "
    "to /content/drive/MyDrive/conversational_training_data.jsonl"
)

with open("/content/drive/MyDrive/conversational_test_data.jsonl", "w", encoding="utf-8") as f:
    for rec in instruction_records_test:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(
    f"\nSaved {len(instruction_records_test)} instruction-tuning records "
    "to /content/drive/MyDrive/conversational_test_data.jsonl"
)
```


#### Your implementation


In [ ]:
# Add your code to format the prompt-response pairs here.

# Add your code here to save your data splits to Google Drive.


## Summary

In the first part of this capstone pathway, you prepared the dataset so that you can use it for fine-tuning an LLM to act as a conversational model. This involved loading the dataset, cleaning the data, splitting it into a training, and test split, and formatting it to be used for fine-tuning.

In the second part of this capstone pathway, you will use these preprocessed data splits to train and evaluate your conversational model.